In [1]:
# Cell 0: Write all patches to disk. Run once → Runtime → Restart session.
# After restart: skip Cell 0, run Cell 1 onward.

import pathlib, site, subprocess, sys

SP = pathlib.Path(site.getsitepackages()[0])
print(f"site-packages: {SP}")

# ── Install packages first so files exist to patch ───────────────────────────
print("📦 Installing packages…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "coqui-tts", "soundfile", "librosa", "scipy",
                "openai-whisper", "jiwer"], check=True)
print("✅ Packages installed")

# ── Patch 1: transformers/pytorch_utils.py → isin_mps_friendly ───────────────
pt_file = SP / "transformers" / "pytorch_utils.py"
src = pt_file.read_text(encoding="utf-8")
PATCH_1 = '''
# >>> PATCHED: isin_mps_friendly for coqui-tts compat <
import torch as _torch
def isin_mps_friendly(elements, test_elements):
    if elements.dtype != test_elements.dtype:
        test_elements = test_elements.to(dtype=elements.dtype)
    return _torch.isin(elements, test_elements)
# >>> END PATCH <
'''
if "isin_mps_friendly" not in src:
    pt_file.write_text(src + PATCH_1, encoding="utf-8")
    print("✅ Patched transformers/pytorch_utils.py")
else:
    print("ℹ️  transformers/pytorch_utils.py already patched")

# ── Patch 2: datasets/features/_torchcodec.py → soundfile-backed AudioDecoder ─
tc_file = SP / "datasets" / "features" / "_torchcodec.py"
PATCH_2 = '''# patched by Colab setup — soundfile-backed AudioDecoder
import io
import numpy as np
import soundfile as _sf


class _Metadata:
    def __init__(self, sample_rate=16000):
        self.path            = None
        self.sample_rate     = sample_rate
        self.num_frames      = 0
        self.num_channels    = 1
        self.bits_per_sample = 16
        self.encoding        = "PCM_S"


class _SampleList:
    def __init__(self, array, sample_rate):
        self.data        = array
        self.sample_rate = sample_rate


class AudioDecoder:
    def __init__(self, source, stream_index=0, sample_rate=None, **kwargs):
        if isinstance(source, (bytes, bytearray, memoryview)):
            buf = io.BytesIO(bytes(source))
        elif isinstance(source, io.IOBase):
            buf = source
        elif source is None:
            buf = None
        else:
            buf = str(source)
        if buf is not None:
            try:
                data, sr = _sf.read(buf, dtype="float32", always_2d=False)
            except Exception:
                data, sr = np.zeros(1600, dtype=np.float32), 16000
        else:
            data, sr = np.zeros(1600, dtype=np.float32), 16000
        if data.ndim > 1:
            data = data.mean(axis=1)
        if sample_rate is not None and sample_rate != sr:
            try:
                import scipy.signal as _sig
                n    = int(len(data) * sample_rate / sr)
                data = _sig.resample(data, n).astype(np.float32)
                sr   = sample_rate
            except Exception:
                pass
        self._samples    = _SampleList(data, sr)
        self._hf_encoded = {}
        self.metadata    = _Metadata(sample_rate=sr)
        self.metadata.num_frames   = len(data)
        self.metadata.num_channels = 1

    def get_all_samples(self):
        return self._samples

_AudioDecoder = AudioDecoder
'''
tc_file.write_text(PATCH_2, encoding="utf-8")
print("✅ Patched datasets/features/_torchcodec.py")

# ── Patch 3: huggingface_hub/utils/_auth.py → _save_stored_tokens stub ────────
hf_auth  = SP / "huggingface_hub" / "utils" / "_auth.py"
auth_src = hf_auth.read_text(encoding="utf-8")
PATCH_3  = '''
# >>> PATCHED: _save_stored_tokens stub <
if "_save_stored_tokens" not in dir():
    def _save_stored_tokens(tokens: dict) -> None:
        pass
# >>> END PATCH <
'''
if "_save_stored_tokens" not in auth_src:
    hf_auth.write_text(auth_src + PATCH_3, encoding="utf-8")
    print("✅ Patched huggingface_hub/utils/_auth.py")
else:
    print("ℹ️  huggingface_hub/utils/_auth.py already patched")

# ── Patch 4: TTS/tts/models/xtts.py → replace load_audio with soundfile ───────
xtts_file = SP / "TTS" / "tts" / "models" / "xtts.py"
xtts_src  = xtts_file.read_text(encoding="utf-8")

OLD_LOAD = """def load_audio(audiopath, sampling_rate):
    # better load setting following: https://github.com/faroit/python_audio_loading_benchmark

    # torchaudio should chose proper backend to load audio depending on platform
    audio, lsr = torchaudio.load(audiopath)

    # stereo to mono if needed
    if audio.size(0) != 1:
        audio = torch.mean(audio, dim=0, keepdim=True)

    if lsr != sampling_rate:
        audio = torchaudio.functional.resample(audio, lsr, sampling_rate)

    # Check some assumptions about audio range. This should be automatically fixed in load_wav_to_torch, but might not be in some edge cases, where we should squawk.
    # '10' is arbitrarily chosen since it seems like audio will often "overdrive" the [-1,1] bounds.
    if torch.any(audio > 10) or not torch.any(audio < 0):
        logger.error("Error with %s. Max=%.2f min=%.2f", audiopath, audio.max(), audio.min())
    # clip audio invalid values
    audio.clip_(-1, 1)
    return audio"""

NEW_LOAD = """def load_audio(audiopath, sampling_rate):
    # PATCHED: soundfile + scipy — avoids torchaudio/AudioDecoder stub clash
    import soundfile as _sf
    import numpy as _np
    import scipy.signal as _sig
    data, sr = _sf.read(str(audiopath), dtype="float32", always_2d=False)
    if data.ndim > 1:
        data = data.mean(axis=1)
    if sr != sampling_rate:
        n    = int(len(data) * sampling_rate / sr)
        data = _sig.resample(data, n).astype(_np.float32)
    data = _np.clip(data, -1.0, 1.0)
    return torch.from_numpy(data).unsqueeze(0)"""

if "PATCHED: soundfile + scipy" not in xtts_src:
    if OLD_LOAD in xtts_src:
        xtts_src = xtts_src.replace(OLD_LOAD, NEW_LOAD)
        xtts_file.write_text(xtts_src, encoding="utf-8")
        print("✅ Patched TTS/tts/models/xtts.py (exact match)")
    else:
        import re
        xtts_src, n = re.subn(
            r"def load_audio\(audiopath, sampling_rate\):[\s\S]+?audio\.clip_\(-1, 1\)\n    return audio",
            NEW_LOAD, xtts_src)
        if n:
            xtts_file.write_text(xtts_src, encoding="utf-8")
            print("✅ Patched TTS/tts/models/xtts.py (regex)")
        else:
            print("⚠️  xtts.py not patched on disk — live injection in Cell 1 will handle it")
else:
    print("ℹ️  TTS/tts/models/xtts.py already patched")

print("\n" + "="*60)
print("ALL PATCHES WRITTEN.")
print("→ Runtime → Restart session")
print("After restart: skip Cell 0, run Cell 1 onward.")
print("="*60)

site-packages: /usr/local/lib/python3.12/dist-packages
📦 Installing packages…
✅ Packages installed
✅ Patched transformers/pytorch_utils.py
✅ Patched datasets/features/_torchcodec.py
ℹ️  huggingface_hub/utils/_auth.py already patched
✅ Patched TTS/tts/models/xtts.py (exact match)

ALL PATCHES WRITTEN.
→ Runtime → Restart session
After restart: skip Cell 0, run Cell 1 onward.


In [1]:
# Cell 1: Runtime patches — run immediately after restart.

import sys, types, importlib.abc, importlib.machinery, torch

# ── torchcodec MetaPathFinder ─────────────────────────────────────────────────
_TC_NAMES = {"torchcodec", "torchcodec.decoders", "torchcodec.decoders._video_decoder"}

class _TCBlocker(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path, target=None):
        if fullname in _TC_NAMES:
            return importlib.machinery.ModuleSpec(
                name=fullname, loader=self,
                is_package=(fullname in {"torchcodec", "torchcodec.decoders"}))
        return None
    def create_module(self, spec):
        if spec.name in sys.modules:
            return sys.modules[spec.name]
        mod = types.ModuleType(spec.name)
        mod.__spec__    = spec
        mod.__loader__  = self
        mod.__package__ = spec.parent
        mod.__path__    = [] if spec.submodule_search_locations is not None else None
        mod.__file__    = None
        return mod
    def exec_module(self, module):
        sys.modules[module.__spec__.name] = module

sys.meta_path = [f for f in sys.meta_path if not isinstance(f, _TCBlocker)]
sys.meta_path.insert(0, _TCBlocker())
import importlib as _il
for _n in list(_TC_NAMES):
    sys.modules.pop(_n, None)
    _il.import_module(_n)

from datasets.features._torchcodec import AudioDecoder as _AD
sys.modules["torchcodec"].decoders = sys.modules["torchcodec.decoders"]
sys.modules["torchcodec.decoders"]._video_decoder = \
    sys.modules["torchcodec.decoders._video_decoder"]
sys.modules["torchcodec.decoders"].AudioDecoder = _AD
print("✅ torchcodec MetaPathFinder + AudioDecoder wired")

from transformers.pytorch_utils import isin_mps_friendly
print("✅ isin_mps_friendly importable")

from huggingface_hub import login
print("✅ huggingface_hub importable")

from TTS.api import TTS
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
print("✅ TTS imported")

# Live load_audio patch (backup if disk patch missed)
import soundfile as _sf_la, numpy as _np_la, scipy.signal as _sig_la

def _load_audio_sf(audiopath, sampling_rate):
    data, sr = _sf_la.read(str(audiopath), dtype="float32", always_2d=False)
    if data.ndim > 1:
        data = data.mean(axis=1)
    if sr != sampling_rate:
        n    = int(len(data) * sampling_rate / sr)
        data = _sig_la.resample(data, n).astype(_np_la.float32)
    data = _np_la.clip(data, -1.0, 1.0)
    return torch.from_numpy(data).unsqueeze(0)

import TTS.tts.models.xtts as _xtts_mod
_xtts_mod.load_audio = _load_audio_sf
print("✅ load_audio patched in module namespace")
print("\n✅ Cell 1 complete — proceed to Cell 2")

✅ torchcodec MetaPathFinder + AudioDecoder wired
✅ isin_mps_friendly importable
✅ huggingface_hub importable
✅ TTS imported
✅ load_audio patched in module namespace

✅ Cell 1 complete — proceed to Cell 2


In [2]:
# Cell 2: HuggingFace login
from huggingface_hub import login
login()

In [3]:
# Cell 3: Load Oshara/xtts-v2-nepali

import os, json, types, torch
import soundfile as _sf_la, numpy as _np_la, scipy.signal as _sig_la
from huggingface_hub import snapshot_download
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
import TTS.tts.models.xtts as _xtts_mod

print("⬇️  Downloading Oshara/xtts-v2-nepali…")
model_dir = snapshot_download(
    "Oshara/xtts-v2-nepali",
    allow_patterns=["epoch-10/*"],
    cache_dir="/content/oshara_xtts"
)
checkpoint_dir = os.path.join(model_dir, "epoch-10")
config_path    = os.path.join(checkpoint_dir, "config.json")

with open(config_path, "r", encoding="utf-8") as f:
    cfg_data = json.load(f)
if "languages" in cfg_data and "ne" not in cfg_data["languages"]:
    cfg_data["languages"].append("ne")
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(cfg_data, f, ensure_ascii=False, indent=2)
print(f"✅ Languages: {cfg_data.get('languages', 'n/a')}")

config = XttsConfig()
config.load_json(config_path)
model  = Xtts.init_from_config(config)
model.load_checkpoint(
    config,
    checkpoint_path=os.path.join(checkpoint_dir, "model.pth"),
    vocab_path=os.path.join(checkpoint_dir, "vocab.json"),
    speaker_file_path=os.path.join(checkpoint_dir, "speakers_xtts.pth"),
    eval=True,
)
model.cuda()

# Inject load_audio into every globals dict the model methods use
def _load_audio_sf(audiopath, sampling_rate):
    data, sr = _sf_la.read(str(audiopath), dtype="float32", always_2d=False)
    if data.ndim > 1:
        data = data.mean(axis=1)
    if sr != sampling_rate:
        n    = int(len(data) * sampling_rate / sr)
        data = _sig_la.resample(data, n).astype(_np_la.float32)
    data = _np_la.clip(data, -1.0, 1.0)
    return torch.from_numpy(data).unsqueeze(0)

for attr_name in dir(model):
    try:
        attr = getattr(model, attr_name)
        if isinstance(attr, types.MethodType):
            g = attr.__func__.__globals__
            if "load_audio" in g or "xtts" in str(g.get("__file__", "")):
                g["load_audio"] = _load_audio_sf
    except Exception:
        pass
model.get_conditioning_latents.__func__.__globals__["load_audio"] = _load_audio_sf
model._clone_voice.__func__.__globals__["load_audio"]             = _load_audio_sf
_xtts_mod.load_audio = _load_audio_sf
print("✅ load_audio injected into all model globals")
print("✅ Oshara XTTS-v2 loaded on GPU")

⬇️  Downloading Oshara/xtts-v2-nepali…


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Languages: ['en', 'es', 'fr', 'de', 'it', 'pt', 'pl', 'tr', 'ru', 'nl', 'cs', 'ar', 'zh-cn', 'hu', 'ko', 'ja', 'hi', 'ne']


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 255), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 255), got 50256. This may result in unexpected behavior.
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 607), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 607), got 50256. This may result in unexpected behavior.


✅ load_audio injected into all model globals
✅ Oshara XTTS-v2 loaded on GPU


In [4]:
# Cell 4: Stream IndicVoices_R Hindi for reference voices, build train/test split

import os, json, random
import numpy as np, soundfile as sf, scipy.signal
from datasets import load_dataset

OUT_DIR     = "/content/xtts_nepali_eval"
SEED        = 42
COLLECT_MAX = 15
SCAN_LIMIT  = 4000
TARGET_SR   = 22050
random.seed(SEED)

for d in ["reference", "ground_truth/male", "ground_truth/female",
          "generated/male", "generated/female"]:
    os.makedirs(f"{OUT_DIR}/{d}", exist_ok=True)

def extract_audio(row):
    audio = row["audio"]
    if isinstance(audio, dict):
        arr = np.array(audio["array"], dtype=np.float32)
        sr  = int(audio["sampling_rate"])
    else:
        s   = audio.get_all_samples()
        arr = np.array(s.data, dtype=np.float32)
        sr  = int(s.sample_rate)
    if arr.ndim > 1:
        arr = arr.mean(axis=1)
    return arr, sr

def resample(arr, src, tgt):
    if src == tgt: return arr
    return scipy.signal.resample(arr, int(len(arr)*tgt/src)).astype(np.float32)

def save_wav(arr, sr, path):
    peak = np.abs(arr).max()
    if peak > 0: arr = arr / peak * 0.95
    sf.write(path, arr, sr)

# Pass 1: scan for speakers
print("🔄 Streaming IndicVoices_R Hindi — Pass 1 (scan)…")
ds1 = load_dataset("ai4bharat/indicvoices_r", "Hindi",
                   split="train", streaming=True)
buckets = {}
for i, row in enumerate(ds1):
    if i >= SCAN_LIMIT: break
    sid    = row.get("speaker_id", "")
    gender = row.get("gender", "").strip().lower()
    if not sid or gender not in ("male", "female"): continue
    if sid not in buckets:
        buckets[sid] = {"gender": gender, "count": 0}
    buckets[sid]["count"] += 1

male_cands   = sorted([(s,v["count"]) for s,v in buckets.items()
                        if v["gender"]=="male"],   key=lambda x:x[1], reverse=True)
female_cands = sorted([(s,v["count"]) for s,v in buckets.items()
                        if v["gender"]=="female"], key=lambda x:x[1], reverse=True)

chosen_male_id   = male_cands[0][0]
chosen_female_id = female_cands[0][0]
print(f"✅ Male   speaker: {chosen_male_id}  ({male_cands[0][1]} utts)")
print(f"✅ Female speaker: {chosen_female_id} ({female_cands[0][1]} utts)")

# Pass 2: collect utterances
print("🔄 Pass 2 (collect)…")
ds2 = load_dataset("ai4bharat/indicvoices_r", "Hindi",
                   split="train", streaming=True)
male_utts, female_utts = [], []
for row in ds2:
    sid    = row.get("speaker_id","")
    gender = row.get("gender","").strip().lower()
    if sid == chosen_male_id and len(male_utts) < COLLECT_MAX:
        arr, sr = extract_audio(row)
        male_utts.append({"array": arr, "sr": sr, "text": row["text"]})
    elif sid == chosen_female_id and len(female_utts) < COLLECT_MAX:
        arr, sr = extract_audio(row)
        female_utts.append({"array": arr, "sr": sr, "text": row["text"]})
    if len(male_utts) >= COLLECT_MAX and len(female_utts) >= COLLECT_MAX:
        break
print(f"✅ Collected {len(male_utts)} male, {len(female_utts)} female utterances")

# Train/test split
def split_utts(utts, n_train=10, n_test=5):
    random.shuffle(utts)
    return utts[:n_train], utts[n_train:n_train+n_test]

male_train,   male_test   = split_utts(male_utts)
female_train, female_test = split_utts(female_utts)

# Save ground-truth test WAVs
for i, utt in enumerate(male_test):
    save_wav(resample(utt["array"], utt["sr"], TARGET_SR),
             TARGET_SR, f"{OUT_DIR}/ground_truth/male/gt_{i+1:02d}.wav")
for i, utt in enumerate(female_test):
    save_wav(resample(utt["array"], utt["sr"], TARGET_SR),
             TARGET_SR, f"{OUT_DIR}/ground_truth/female/gt_{i+1:02d}.wav")

# Build reference WAVs
def build_ref(utts, path):
    silence = np.zeros(int(TARGET_SR*0.3), dtype=np.float32)
    parts   = []
    for u in utts:
        parts.append(resample(u["array"], u["sr"], TARGET_SR))
        parts.append(silence)
    combined = np.concatenate(parts)[:TARGET_SR*30]
    save_wav(combined, TARGET_SR, path)
    print(f"  {os.path.basename(path)}  ({len(combined)/TARGET_SR:.1f}s)")

print("Building reference WAVs…")
male_ref_path   = f"{OUT_DIR}/reference/male_reference.wav"
female_ref_path = f"{OUT_DIR}/reference/female_reference.wav"
build_ref(male_train,   male_ref_path)
build_ref(female_train, female_ref_path)

meta = {
    "male_speaker_id":    chosen_male_id,
    "female_speaker_id":  chosen_female_id,
    "male_test_texts":    [u["text"] for u in male_test],
    "female_test_texts":  [u["text"] for u in female_test],
}
with open(f"{OUT_DIR}/speaker_metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"✅ Reference WAVs and train/test split complete")

🔄 Streaming IndicVoices_R Hindi — Pass 1 (scan)…


README.md:   0%|          | 0.00/33.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/246 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/99 [00:00<?, ?it/s]

✅ Male   speaker: S4259588400343717  (69 utts)
✅ Female speaker: S4259569100343205 (63 utts)
🔄 Pass 2 (collect)…


Resolving data files:   0%|          | 0/246 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/99 [00:00<?, ?it/s]

✅ Collected 15 male, 15 female utterances
Building reference WAVs…
  male_reference.wav  (30.0s)
  female_reference.wav  (30.0s)
✅ Reference WAVs and train/test split complete


In [10]:
# Fix: patch tokenizer.preprocess_text to treat "ne" same as "hi" (both Devanagari)

import site, pathlib

SP = pathlib.Path(site.getsitepackages()[0])
tok_file = SP / "TTS" / "tts" / "layers" / "xtts" / "tokenizer.py"
src = tok_file.read_text(encoding="utf-8")

# Find and patch preprocess_text to add "ne" → treated as "hi"
OLD_PREPROCESS = '''    def preprocess_text(self, txt, lang):'''

# We'll monkey-patch the method directly on the tokenizer instance
# since disk patch requires restart

import types, re

original_preprocess = model.tokenizer.preprocess_text.__func__

def patched_preprocess_text(self, txt, lang):
    # Remap Nepali to Hindi — same Devanagari script, same preprocessing
    if lang == "ne":
        lang = "hi"
    return original_preprocess(self, txt, lang)

model.tokenizer.preprocess_text = types.MethodType(
    patched_preprocess_text, model.tokenizer
)
print("✅ tokenizer.preprocess_text patched: 'ne' → 'hi'")

# Also patch tokenizer.encode in case it has its own lang check
original_encode = model.tokenizer.encode.__func__

def patched_encode(self, txt, lang):
    if lang == "ne":
        lang = "hi"
    return original_encode(self, txt, lang)

model.tokenizer.encode = types.MethodType(patched_encode, model.tokenizer)
print("✅ tokenizer.encode patched: 'ne' → 'hi'")

# Also patch char_limits
if "ne" not in model.tokenizer.char_limits:
    model.tokenizer.char_limits["ne"] = model.tokenizer.char_limits.get("hi", 150)
    print(f"✅ char_limits['ne'] = {model.tokenizer.char_limits['ne']}")

# Also patch inference() in xtts.py globals to remap ne→hi for split_sentence call
# by patching split_sentence itself
from TTS.tts.layers.xtts.tokenizer import split_sentence as _orig_split

def _patched_split(text, lang, char_limit):
    if lang == "ne":
        lang = "hi"
    return _orig_split(text, lang, char_limit)

# Inject into the globals of the inference method
model.inference.__func__.__globals__["split_sentence"] = _patched_split
print("✅ split_sentence patched in inference globals: 'ne' → 'hi'")

# Test
import tempfile, soundfile as sf, numpy as np, scipy.signal, traceback

data, sr = sf.read("/content/xtts_nepali_eval/reference/male_reference.wav",
                   dtype="float32", always_2d=False)
if data.ndim > 1: data = data.mean(axis=1)
if sr != 22050:
    data = scipy.signal.resample(data, int(len(data)*22050/sr)).astype("float32")
tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
sf.write(tmp.name, data, 22050)

try:
    out = model.synthesize(
        "कागती खाएर केटाकेटीहरू खुसी हुँदै घर गए।",
        config, speaker_wav=tmp.name, language="ne",
        temperature=0.65, repetition_penalty=5.0,
        top_k=50, top_p=0.85, speed=1.0, enable_text_splitting=True,
    )
    wav = np.array(out["wav"], dtype=np.float32)
    sf.write("/content/test_nep.wav", wav, 24000)
    print(f"✅ SUCCESS — shape: {wav.shape}")
except Exception:
    traceback.print_exc()

✅ tokenizer.preprocess_text patched: 'ne' → 'hi'
✅ tokenizer.encode patched: 'ne' → 'hi'
✅ split_sentence patched in inference globals: 'ne' → 'hi'
✅ SUCCESS — shape: (99072,)


In [11]:
# Cell 5: Generate Nepali evaluation set

import os, json, tempfile, types, torch
import numpy as np, soundfile as sf, scipy.signal
import soundfile as _sf_la, numpy as _np_la, scipy.signal as _sig_la
import TTS.tts.models.xtts as _xtts_mod

OUT_DIR = "/content/xtts_nepali_eval"

# Add to top of Cell 5 — patch tokenizer for Nepali support
import types
from TTS.tts.layers.xtts.tokenizer import split_sentence as _orig_split

# char_limits
if "ne" not in model.tokenizer.char_limits:
    model.tokenizer.char_limits["ne"] = model.tokenizer.char_limits.get("hi", 150)

# preprocess_text
_orig_pre = model.tokenizer.preprocess_text.__func__
def _pre(self, txt, lang):
    return _orig_pre(self, txt, "hi" if lang == "ne" else lang)
model.tokenizer.preprocess_text = types.MethodType(_pre, model.tokenizer)

# encode
_orig_enc = model.tokenizer.encode.__func__
def _enc(self, txt, lang):
    return _orig_enc(self, txt, "hi" if lang == "ne" else lang)
model.tokenizer.encode = types.MethodType(_enc, model.tokenizer)

# split_sentence in inference globals
def _split(text, lang, char_limit):
    return _orig_split(text, "hi" if lang == "ne" else lang, char_limit)
model.inference.__func__.__globals__["split_sentence"] = _split

print("✅ Tokenizer fully patched for Nepali")


# Re-apply load_audio patch (always do this before synthesis)
def _load_audio_sf(audiopath, sampling_rate):
    data, sr = _sf_la.read(str(audiopath), dtype="float32", always_2d=False)
    if data.ndim > 1:
        data = data.mean(axis=1)
    if sr != sampling_rate:
        n    = int(len(data) * sampling_rate / sr)
        data = _sig_la.resample(data, n).astype(_np_la.float32)
    data = _np_la.clip(data, -1.0, 1.0)
    return torch.from_numpy(data).unsqueeze(0)

for attr_name in dir(model):
    try:
        attr = getattr(model, attr_name)
        if isinstance(attr, types.MethodType):
            g = attr.__func__.__globals__
            if "load_audio" in g or "xtts" in str(g.get("__file__", "")):
                g["load_audio"] = _load_audio_sf
    except Exception:
        pass
model.get_conditioning_latents.__func__.__globals__["load_audio"] = _load_audio_sf
model._clone_voice.__func__.__globals__["load_audio"]             = _load_audio_sf
_xtts_mod.load_audio = _load_audio_sf
print("✅ load_audio re-patched")

# Eval set
EVAL_NEPALI = {
    "NEP_01": "कागती खाएर केटाकेटीहरू खुसी हुँदै घर गए।",
    "NEP_02": "चराहरू चिरबिर गर्दै चाँडै उडेर चौतारीमा बसे।",
    "NEP_03": "ठूलो डाँडामाथि ढकमक्क गुराँस फुल्दा धेरै राम्रो देखिन्छ।",
    "NEP_04": "फराकिलो बाटोमा भाइ र बहिनी पानी पिउँदै हिँडे।",
    "NEP_05": "शहरको शान्त सडकमा हिजो बेलुका हुरी चल्यो।",
    "NEP_06": "मेरो नयाँ घरमा जताततै घामको न्यानो उज्यालो आउँछ।",
    "NEP_07": "ज्ञान र विज्ञानको क्षेत्रमा प्रगति गर्न निरन्तर मिहिनेत आवश्यक छ।",
    "NEP_08": "किसानले खेतमा धान रोपेर मुरी फलाउने आशा राखेका छन्।",
    "NEP_09": "चाडपर्वमा मान्यजनको हातबाट टीका र जमरा थाप्नु हाम्रो परम्परा हो।",
    "NEP_10": "वर्षायाममा खोलानाला बढेर बाढी आउने खतरा सधैँ रहन्छ।",
    "NEP_11": "शिक्षकले कक्षामा गणित र व्याकरणका कठिन प्रश्नहरू सोध्नुभयो।",
    "NEP_12": "आमाले बिहानै उठेर मीठो सेलरोटी र तरकारी पकाउनुभयो।",
    "NEP_13": "जङ्गलमा बाघ, भालु, गैँडा र हात्तीजस्ता जङ्गली जनावरहरू पाइन्छन्।",
    "NEP_14": "दुःख र सुख जीवनका दुई पाटा हुन्, त्यसैले धैर्य गर्नुपर्छ।",
    "NEP_15": "स्वास्थ्य नै धन हो, त्यसैले सन्तुलित भोजन र व्यायाममा ध्यान दिनुपर्छ।",
    "NEP_16": "आजकल मोबाइल र इन्टरनेटको माध्यमबाट संसारभरको खबर क्षणभरमै सुन्न सकिन्छ।",
    "NEP_17": "बजारमा तरकारी र फलफूलको भाउ एक्कासि बढेर ग्राहकहरू मारमा परेका छन्।",
    "NEP_18": "लोकगीत र बाँसुरीको धुनले गाउँको रातलाई झनै रमाइलो बनाउँछ।",
    "NEP_19": "सगरमाथाको चुचुरोमा पुग्न हिउँ र चिसो हावासँग सङ्घर्ष गर्नुपर्छ।",
    "NEP_20": "यो लामो बाटो पार गर्न युवाहरूलाई एकदमै रमाइलो लाग्छ।",
}

def prepare_ref(path, target_sr=22050):
    data, sr = sf.read(path, dtype="float32", always_2d=False)
    if data.ndim > 1: data = data.mean(axis=1)
    if sr != target_sr:
        data = scipy.signal.resample(data, int(len(data)*target_sr/sr)).astype("float32")
    peak = np.abs(data).max()
    if peak > 0: data = data / peak * 0.95
    t = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    sf.write(t.name, data, target_sr)
    t.close()
    return t.name

male_ref_tmp   = prepare_ref(f"{OUT_DIR}/reference/male_reference.wav")
female_ref_tmp = prepare_ref(f"{OUT_DIR}/reference/female_reference.wav")
print("✅ Reference WAVs prepared")

SYNTH_KWARGS = dict(
    language="ne",
    temperature=0.65,
    repetition_penalty=5.0,
    top_k=50, top_p=0.85,
    speed=1.0,
    enable_text_splitting=True,
)
SAMPLE_RATE = 24000

gen_log = []
for gender, ref_tmp in [("male", male_ref_tmp), ("female", female_ref_tmp)]:
    print(f"\n🎙️  Generating Nepali {gender}…")
    for sid, text in EVAL_NEPALI.items():
        out_path = f"{OUT_DIR}/generated/{gender}/{sid}_{gender}.wav"
        print(f"  [{sid}]", end="  ")
        try:
            out  = model.synthesize(text, config, speaker_wav=ref_tmp, **SYNTH_KWARGS)
            wav  = np.array(out["wav"], dtype=np.float32)
            peak = np.abs(wav).max()
            if peak > 0: wav = wav / peak * 0.95
            sf.write(out_path, wav, SAMPLE_RATE)
            print("✅")
            gen_log.append({"id": sid, "gender": gender, "status": "ok", "path": out_path})
        except Exception as e:
            print(f"❌ {e}")
            gen_log.append({"id": sid, "gender": gender, "status": f"error:{e}", "path": None})

for p in [male_ref_tmp, female_ref_tmp]:
    try: os.unlink(p)
    except: pass

with open(f"{OUT_DIR}/generation_log.json", "w", encoding="utf-8") as f:
    json.dump(gen_log, f, ensure_ascii=False, indent=2)

ok = sum(1 for r in gen_log if r["status"] == "ok")
print(f"\n✅ Generation: {ok}/{len(gen_log)} succeeded")

✅ Tokenizer fully patched for Nepali
✅ load_audio re-patched
✅ Reference WAVs prepared

🎙️  Generating Nepali male…
  [NEP_01]  ✅
  [NEP_02]  ✅
  [NEP_03]  ✅
  [NEP_04]  ✅
  [NEP_05]  ✅
  [NEP_06]  ✅
  [NEP_07]  ✅
  [NEP_08]  ✅
  [NEP_09]  ✅
  [NEP_10]  ✅
  [NEP_11]  ✅
  [NEP_12]  ✅
  [NEP_13]  ✅
  [NEP_14]  ✅
  [NEP_15]  ✅
  [NEP_16]  ✅
  [NEP_17]  ✅
  [NEP_18]  ✅
  [NEP_19]  ✅
  [NEP_20]  ✅

🎙️  Generating Nepali female…
  [NEP_01]  ✅
  [NEP_02]  ✅
  [NEP_03]  ✅
  [NEP_04]  ✅
  [NEP_05]  ✅
  [NEP_06]  ✅
  [NEP_07]  ✅
  [NEP_08]  ✅
  [NEP_09]  ✅
  [NEP_10]  ✅
  [NEP_11]  ✅
  [NEP_12]  ✅
  [NEP_13]  ✅
  [NEP_14]  ✅
  [NEP_15]  ✅
  [NEP_16]  ✅
  [NEP_17]  ✅
  [NEP_18]  ✅
  [NEP_19]  ✅
  [NEP_20]  ✅

✅ Generation: 40/40 succeeded


In [12]:
# Cell 6: WER/CER table for Nepali using Whisper medium

import os, re, json, torch, whisper
from jiwer import wer, cer

OUT_DIR = "/content/xtts_nepali_eval"

try:
    asr
    print("✅ Whisper already loaded")
except NameError:
    print("🔄 Loading Whisper medium…")
    asr = whisper.load_model("medium", device="cuda" if torch.cuda.is_available() else "cpu")
    print("✅ Whisper medium loaded")

def normalise(text):
    text = re.sub(r"[।॥,.!?;:\"'()\-]", " ", text.strip())
    return re.sub(r"\s+", " ", text).strip()

def score(wav_path, ref_text):
    result = asr.transcribe(wav_path, language="ne", task="transcribe",
                             fp16=torch.cuda.is_available())
    hyp = result["text"].strip()
    w   = wer(normalise(ref_text), normalise(hyp))
    c   = cer(normalise(ref_text), normalise(hyp))
    return round(w, 4), round(c, 4)

NEPALI_META = [
    ("NEP_01", "velars_gutturals",              "कागती खाएर केटाकेटीहरू खुसी हुँदै घर गए।"),
    ("NEP_02", "palatals_and_trills",           "चराहरू चिरबिर गर्दै चाँडै उडेर चौतारीमा बसे।"),
    ("NEP_03", "retroflexes_nasalization",      "ठूलो डाँडामाथि ढकमक्क गुराँस फुल्दा धेरै राम्रो देखिन्छ।"),
    ("NEP_04", "labials_and_nasalization",      "फराकिलो बाटोमा भाइ र बहिनी पानी पिउँदै हिँडे।"),
    ("NEP_05", "fricatives_and_glides",         "शहरको शान्त सडकमा हिजो बेलुका हुरी चल्यो।"),
    ("NEP_06", "nasals_and_conjuncts",          "मेरो नयाँ घरमा जताततै घामको न्यानो उज्यालो आउँछ।"),
    ("NEP_07", "complex_conjuncts",             "ज्ञान र विज्ञानको क्षेत्रमा प्रगति गर्न निरन्तर मिहिनेत आवश्यक छ।"),
    ("NEP_08", "dentals_vowel_lengths",         "किसानले खेतमा धान रोपेर मुरी फलाउने आशा राखेका छन्।"),
    ("NEP_09", "affricates_labiodentals",       "चाडपर्वमा मान्यजनको हातबाट टीका र जमरा थाप्नु हाम्रो परम्परा हो।"),
    ("NEP_10", "retroflex_flaps",               "वर्षायाममा खोलानाला बढेर बाढी आउने खतरा सधैँ रहन्छ।"),
    ("NEP_11", "retroflex_nasals_clusters",     "शिक्षकले कक्षामा गणित र व्याकरणका कठिन प्रश्नहरू सोध्नुभयो।"),
    ("NEP_12", "unaspirated_aspirated_stops",   "आमाले बिहानै उठेर मीठो सेलरोटी र तरकारी पकाउनुभयो।"),
    ("NEP_13", "velar_nasals_heavy_aspirates",  "जङ्गलमा बाघ, भालु, गैँडा र हात्तीजस्ता जङ्गली जनावरहरू पाइन्छन्।"),
    ("NEP_14", "visarga_and_glides",            "दुःख र सुख जीवनका दुई पाटा हुन्, त्यसैले धैर्य गर्नुपर्छ।"),
    ("NEP_15", "dense_consonant_clusters",      "स्वास्थ्य नै धन हो, त्यसैले सन्तुलित भोजन र व्यायाममा ध्यान दिनुपर्छ।"),
    ("NEP_16", "loan_words_retroflexes",        "आजकल मोबाइल र इन्टरनेटको माध्यमबाट संसारभरको खबर क्षणभरमै सुन्न सकिन्छ।"),
    ("NEP_17", "geminates_and_labials",         "बजारमा तरकारी र फलफूलको भाउ एक्कासि बढेर ग्राहकहरू मारमा परेका छन्।"),
    ("NEP_18", "palatal_aspirates_liquids",     "लोकगीत र बाँसुरीको धुनले गाउँको रातलाई झनै रमाइलो बनाउँछ।"),
    ("NEP_19", "alveolar_fricatives_conjuncts", "सगरमाथाको चुचुरोमा पुग्न हिउँ र चिसो हावासँग सङ्घर्ष गर्नुपर्छ।"),
    ("NEP_20", "approximants_and_flow",         "यो लामो बाटो पार गर्न युवाहरूलाई एकदमै रमाइलो लाग्छ।"),
]

print("📊 Transcribing and scoring…")
rows = []
for sid, cat, ref in NEPALI_META:
    row = {"id": sid, "category": cat}
    for gender in ["male", "female"]:
        wp = f"{OUT_DIR}/generated/{gender}/{sid}_{gender}.wav"
        if os.path.exists(wp):
            try:
                w, c = score(wp, ref)
                row[f"{gender}_wer"] = w
                row[f"{gender}_cer"] = c
            except Exception as e:
                row[f"{gender}_wer"] = None
                row[f"{gender}_cer"] = None
                print(f"  ⚠️  {sid} {gender}: {e}")
        else:
            row[f"{gender}_wer"] = None
            row[f"{gender}_cer"] = None
    rows.append(row)
    print(f"  {sid} ✅")

# Print table
print("\n" + "="*90)
print("  OSHARA XTTS-v2 — NEPALI EVALUATION  (Whisper medium, language=ne)")
print("="*90)
print(f"{'ID':<8} {'Category':<38} {'M-WER':>6} {'M-CER':>6} {'F-WER':>6} {'F-CER':>6}")
print("-"*90)
for r in rows:
    mw = f"{r['male_wer']:.3f}"   if r["male_wer"]   is not None else "  N/A"
    mc = f"{r['male_cer']:.3f}"   if r["male_cer"]   is not None else "  N/A"
    fw = f"{r['female_wer']:.3f}" if r["female_wer"] is not None else "  N/A"
    fc = f"{r['female_cer']:.3f}" if r["female_cer"] is not None else "  N/A"
    print(f"{r['id']:<8} {r['category']:<38} {mw:>6} {mc:>6} {fw:>6} {fc:>6}")
print("-"*90)
for gender, wk, ck in [("Male","male_wer","male_cer"), ("Female","female_wer","female_cer")]:
    v = [r for r in rows if r[wk] is not None]
    if v:
        print(f"  AVG {gender:6s}: WER={sum(r[wk] for r in v)/len(v):.3f}  "
              f"CER={sum(r[ck] for r in v)/len(v):.3f}  (n={len(v)})")
print("="*90)

with open(f"{OUT_DIR}/wer_cer_nepali.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print(f"\n✅ Results saved → {OUT_DIR}/wer_cer_nepali.json")

🔄 Loading Whisper medium…


100%|█████████████████████████████████████| 1.42G/1.42G [00:22<00:00, 68.4MiB/s]


✅ Whisper medium loaded
📊 Transcribing and scoring…
  NEP_01 ✅
  NEP_02 ✅
  NEP_03 ✅
  NEP_04 ✅
  NEP_05 ✅
  NEP_06 ✅
  NEP_07 ✅
  NEP_08 ✅
  NEP_09 ✅
  NEP_10 ✅
  NEP_11 ✅
  NEP_12 ✅
  NEP_13 ✅
  NEP_14 ✅
  NEP_15 ✅
  NEP_16 ✅
  NEP_17 ✅
  NEP_18 ✅
  NEP_19 ✅
  NEP_20 ✅

  OSHARA XTTS-v2 — NEPALI EVALUATION  (Whisper medium, language=ne)
ID       Category                                M-WER  M-CER  F-WER  F-CER
------------------------------------------------------------------------------------------
NEP_01   velars_gutturals                        1.143  0.308  1.143  0.308
NEP_02   palatals_and_trills                     1.286  0.372  1.286  0.395
NEP_03   retroflexes_nasalization                1.125  0.364  1.000  0.382
NEP_04   labials_and_nasalization                1.250  0.364  0.875  0.341
NEP_05   fricatives_and_glides                   0.714  0.150  0.857  0.175
NEP_06   nasals_and_conjuncts                    0.750  0.170  0.875  0.213
NEP_07   complex_conjuncts          

In [13]:
# Cell 7: Zip and download
import shutil
from google.colab import files

zip_path = "/content/xtts_nepali_output"
print("📦 Zipping…")
shutil.make_archive(zip_path, "zip", "/content/xtts_nepali_eval")
print("⬇️  Downloading…")
files.download(f"{zip_path}.zip")

📦 Zipping…
⬇️  Downloading…


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>